In [41]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

path = Path('/home/vishal/Waymo/2011_09_26_drive_0005_sync/2011_09_26/2011_09_26_drive_0005_sync')

rel_path = Path('velodyne_points/data')
fin_path = path/rel_path
# Change file
def extract_raw_lidar_data(path):
    data = np.fromfile(path, dtype=np.float32)
    data = data.reshape(-1, 4)
    return data

data = extract_raw_lidar_data(fin_path / '0000000150.bin')
print(data.shape)
x,y,z,intensity = data.T

plt.figure(figsize=(8,8))
plt.scatter(x, y, c=intensity, s=0.5, cmap='viridis', linewidths=0, alpha=0.9)
plt.xlabel('x'); plt.ylabel('y'); plt.title('Top-down (x,y) colored by intensity')
plt.axis('equal')
plt.show()

(115781, 4)


/tmp/ipykernel_342530/2168512144.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [42]:
# Convert LiDAR coordinates to Camera Coordinates
file_path = '2011_09_26_drive_0005_sync/2011_09_26/2011_09_26_drive_0005_sync/2011_09_26/calib_velo_to_cam.txt'

def extract_transform(path):
    data = {}
    with open(path, 'r') as file:
        for line in file:
            key, value = line.split(':',1)
            data[key.strip()] = value.strip()
    R = np.array(data['R'].split(),dtype=np.float64).reshape(3, 3)
    t = np.array(data['T'].split(),dtype=np.float64).reshape(3, 1)

    g = np.eye(4)
    g[:3, :3] = R
    g[:3, 3] = t.flatten()
    return g

g = extract_transform(file_path)
print(g)

[[ 7.533745e-03 -9.999714e-01 -6.166020e-04 -4.069766e-03]
 [ 1.480249e-02  7.280733e-04 -9.998902e-01 -7.631618e-02]
 [ 9.998621e-01  7.523790e-03  1.480755e-02 -2.717806e-01]
 [ 0.000000e+00  0.000000e+00  0.000000e+00  1.000000e+00]]


In [43]:
# Convert all points to camera frame
p_cam_frame = []
p_lidar_frame = np.column_stack([x, y, z, np.ones((len(x),1))])
p_cam_frame = (g @ p_lidar_frame.T).T
p_cam_frame = p_cam_frame[:, :3]
p_cam_frame = p_cam_frame[p_cam_frame[:, 2] > 0]
print(np.shape(p_cam_frame))

(56317, 3)


In [44]:
intrinsics_path ='2011_09_26_drive_0005_sync/2011_09_26/2011_09_26_drive_0005_sync/2011_09_26/calib_cam_to_cam.txt'

def extract_intrinsic_transform(intrinsics_path):
    dict = {}
    with open(intrinsics_path, 'r') as file:
        for lines in file:
            key, value = lines.strip().split(':',1)
            dict[key] = value
    intrinsic_matrix = np.array(dict['P_rect_02'].split(), dtype=np.float32).reshape(3, 4)
    return intrinsic_matrix

intrinsic_tx = extract_intrinsic_transform(intrinsics_path)
print(extract_intrinsic_transform(intrinsics_path))


[[7.215377e+02 0.000000e+00 6.095593e+02 4.485728e+01]
 [0.000000e+00 7.215377e+02 1.728540e+02 2.163791e-01]
 [0.000000e+00 0.000000e+00 1.000000e+00 2.745884e-03]]


In [45]:
translation_mtx = []
#print(len(p_cam_frame.T[0]))
for num in range(len(p_cam_frame.T[0])):
    translation_mtx = np.append(translation_mtx, intrinsic_tx[:,3].T).reshape(-1,3)
#print(np.shape(translation_mtx))
#print(translation_mtx)
# (u', v', z)
final = (intrinsic_tx[:,:3] @ p_cam_frame.T).T + translation_mtx
final = final[final[:,2]>0]  # Remove points with zero or negative depth
final = final[final[:,0]<1242*final[:,2]]
final = final[final[:,1]<375*final[:,2]]
depth = final[:, 2]
u = (final[:, 0] / depth).astype(int)
v = (final[:, 1] / depth).astype(int)
pixels = np.column_stack((u, v))
print(np.shape(final))
# pixels = pixels[pixels[:,0]<1242]
# pixels = pixels[pixels[:,1]<375]
# print(np.shape(pixels))
# print(pixels)

(24422, 3)


In [46]:
# Load and display the image
img = mpimg.imread('2011_09_26_drive_0005_sync/2011_09_26/2011_09_26_drive_0005_sync/image_02/data/0000000150.png')
fig, ax = plt.subplots(figsize=(12, 4))
ax.imshow(img)
ax.scatter(u, v, c=final[:, 2], cmap='turbo', marker='o', s=1, alpha=0.3)
ax.set_xlim(0, img.shape[1])
ax.set_ylim(img.shape[0], 0)
ax.set_aspect('equal')
ax.set_title('LiDAR points projected into image_02')
plt.show()

/tmp/ipykernel_342530/2311546212.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [47]:
# Generate for each frame and convert to a video.
img_folder = Path('2011_09_26_drive_0005_sync/2011_09_26/2011_09_26_drive_0005_sync/image_02/data')
lidar_folder = Path('2011_09_26_drive_0005_sync/2011_09_26/2011_09_26_drive_0005_sync/velodyne_points/data')

data = []
for file_path in sorted(lidar_folder.iterdir(), key=lambda p: p.name):
    arr_2d = extract_raw_lidar_data(file_path)
    data.append(arr_2d[:, :3])

data_final = []
P = intrinsic_tx @ g

for arr_2d in data:
    arr_h = np.column_stack([arr_2d, np.ones(len(arr_2d))]) 
    dummy_proj = (P @ arr_h.T).T
    dummy = dummy_proj[
        (dummy_proj[:, 2] > 0) & 
        (dummy_proj[:, 0] < 1242 * dummy_proj[:, 2]) & 
        (dummy_proj[:, 1] < 375 * dummy_proj[:, 2])
    ]
    data_final.append(dummy)

print(f"Processed {len(data_final)} frames successfully.")


Processed 154 frames successfully.


In [48]:
import cv2
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for saving frames

img_files = sorted(img_folder.iterdir(), key=lambda p: p.name)

# Get frame size from first image
first_img = mpimg.imread(img_files[0])
h, w = first_img.shape[:2]

# Set up video writer
out = cv2.VideoWriter(
    'lidar_projection.mp4',
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps=10,
    frameSize=(w, h)
)

for img_path, lidar_pts in zip(img_files, data_final):
    img = mpimg.imread(img_path)

    # Render frame with matplotlib
    fig, ax = plt.subplots(figsize=(w / 100, h / 100), dpi=100)
    ax.imshow(img)

    depth = lidar_pts[:, 2]
    u = (lidar_pts[:, 0] / depth).astype(int)
    v = (lidar_pts[:, 1] / depth).astype(int)

    ax.scatter(u, v, c=depth, cmap='turbo', s=1, alpha=0.3)
    ax.set_xlim(0, w)
    ax.set_ylim(h, 0)
    ax.axis('off')
    plt.tight_layout(pad=0)

    # Convert matplotlib figure → numpy array → BGR for OpenCV
    fig.canvas.draw()
    frame = np.array(fig.canvas.buffer_rgba())[:, :, :3]  # replaces tostring_rgb
    frame = cv2.resize(frame, (w, h))
    frame_bgr = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)

    out.write(frame_bgr)
    plt.close(fig)

out.release()
print("Saved lidar_projection.mp4")

Saved lidar_projection.mp4
